# Medical knowledge chunking playground

    Use this notebook to experiment with cleaning, sectioning, and chunking medical guideline PDFs (or text exports) before feeding them into the RAG pipeline. Update the paths and chunking parameters to mirror your environment, then inspect how the resulting chunks look before exporting to JSONL.

## Prerequisites
    - Install either `pypdf` or `PyPDF2` so `_extract_pdf_text` can read PDFs.
    - Point `knowledge_dir` to your `Medical_Knowledge` folder (or a directory of `.txt` exports).
    - Optional: install `matplotlib` if you want to visualize chunk length distributions.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import os

# Add the src directory to PYTHONPATH
sys.path.append(os.path.abspath("../src"))

from pathlib import Path
import json
import statistics
import textwrap

from evidence_rl.ingestion import (
    _extract_pdf_text,
    _iter_sections,
    chunk_guideline_text,
    export_documents_jsonl,
)
from evidence_rl.documents import Document

In [3]:
# Path to your clinical guideline PDFs or text exports
knowledge_dir = Path("/home/hice1/jtamo3/bmed-sp-wang/Ben/Data/Medical-Cardiac-Knowledge")

# Chunking configuration you want to experiment with
chunk_size = 400  # words per chunk
overlap = 80     # word overlap between chunks

In [4]:
if not knowledge_dir.exists():
    raise FileNotFoundError(f"Knowledge directory not found: {knowledge_dir}")

files = sorted([p for p in knowledge_dir.glob('**/*') if p.suffix.lower() in {'.txt'}])
print(f'Found {len(files)} files under {knowledge_dir}')
for path in files[:5]:
    print(f' - {path}')

Found 2 files under /home/hice1/jtamo3/bmed-sp-wang/Ben/Data/Medical-Cardiac-Knowledge
 - /home/hice1/jtamo3/bmed-sp-wang/Ben/Data/Medical-Cardiac-Knowledge/1979-nomenclature-and-criteria-for-diagnosis-of-ischemic-heart-disease-report-of-the-joint-international-society-and.txt
 - /home/hice1/jtamo3/bmed-sp-wang/Ben/Data/Medical-Cardiac-Knowledge/armen-et-al-1958-pulmonary-heart-disease.txt


### Peek at raw text from a sample file
    Change `sample_path` if you want to inspect a different file. Trimming keeps the output manageable.

In [6]:
def load_txt(path: str | Path) -> str:
    with open(path, encoding="utf-8") as f:
        return f.read()

def normalize_paragraphs(text: str) -> str:
    """Join wrapped lines inside paragraphs, keep blank lines as paragraph breaks."""
    blocks = text.split("\n\n")
    normalized_blocks = [" ".join(b.splitlines()) for b in blocks]
    return "\n\n".join(normalized_blocks)

In [8]:
sample_path = files[0]
# raw_text = _extract_pdf_text(sample_path)
print(f'Sampled file: {sample_path.name}')
# print(textwrap.shorten(raw_text.replace('', ' '), width=2000, placeholder='...'))

raw_text = load_txt(sample_path)
text = normalize_paragraphs(raw_text)
print(textwrap.shorten(raw_text, width=2000, placeholder='...'))

Sampled file: 1979-nomenclature-and-criteria-for-diagnosis-of-ischemic-heart-disease-report-of-the-joint-international-society-and.txt
Nomenclature and Criteria for Diagnosis of Ischemic Heart Disease Report of the Joint International Society and Federation of Cardiology/World Health Organization Task Force on Standardization of Clinical Nomenclature LONG AGO, EPIDEMIOLOGISTS REALIZED the necessity of standardizing terminology and diagnostic criteria. Today, this need should also be recognized by clinicians. New, expensive medical and surgical methods of treatment are introduced every day, and it is essential to evaluate the effectiveness of these methods reliably and objectively. However, comparison of results is only possible if the object of the evaluation has been defined in a standard manner. The need for agreement on nomenclature is particularly urgent in the field of ischemic heart disease (IHD) because extraordinary advances have been achieved by new methods of diagnosis and tr

### Inspect detected sections
    Section detection uses numbered headings or uppercase titles as heuristics.

In [9]:
sections = list(_iter_sections(raw_text))
print(f'Detected {len(sections)} sections')
for title, body in sections:
    preview = textwrap.shorten(body, width=240, placeholder='...')
    print(f"[{title}]{preview}")

Detected 12 sections
[preamble]preamble Nomenclature and Criteria for Diagnosis of Ischemic Heart Disease Report of the Joint International Society and Federation of Cardiology/World Health Organization Task Force on Standardization of Clinical Nomenclature
[LONG AGO, EPIDEMIOLOGISTS REALIZED]LONG AGO, EPIDEMIOLOGISTS REALIZED the necessity of standardizing terminology and diagnostic criteria. Today, this need should also be recognized by clinicians. New, expensive medical and surgical methods of treatment are introduced...
[0. Ischemic Heart Disease]0. Ischemic Heart Disease IHD is defined as myocardial impairment due to an imbalance between coronary blood flow and myocardial requirements caused by changes in the coronary circulation. IHD comprises acute and temporary as well as...
[1. Primary Cardiac Arrest]1. Primary Cardiac Arrest Primary cardiac arrest is a sudden event, presumably due to electric instability of the heart, where evidence which allows other diagnosis' is lacking. I

### Chunk the guideline text
    Adjust `chunk_size` and `overlap` above to see how the chunking changes.

In [13]:
chunks = chunk_guideline_text(
    raw_text,
    source_id=sample_path,
    chunk_size=chunk_size,
    overlap=overlap,
)
print(f'Generated {len(chunks)} chunks from {sample_path}')
print(f'First chunk metadata: {chunks[0].metadata}')
print(f"First chunk preview: {textwrap.shorten(chunks[0].text, width=280, placeholder='...')}")

Generated 12 chunks from /home/hice1/jtamo3/bmed-sp-wang/Ben/Data/Medical-Cardiac-Knowledge/1979-nomenclature-and-criteria-for-diagnosis-of-ischemic-heart-disease-report-of-the-joint-international-society-and.txt
First chunk metadata: {'source_id': PosixPath('/home/hice1/jtamo3/bmed-sp-wang/Ben/Data/Medical-Cardiac-Knowledge/1979-nomenclature-and-criteria-for-diagnosis-of-ischemic-heart-disease-report-of-the-joint-international-society-and.txt'), 'section_title': 'preamble', 'section_index': 0, 'chunk_index': 0}
First chunk preview: preamble Nomenclature and Criteria for Diagnosis of Ischemic Heart Disease Report of the Joint International Society and Federation of Cardiology/World Health Organization Task Force on Standardization of Clinical Nomenclature


In [17]:
print(chunks[3].text)

Primary Cardiac Arrest Primary cardiac arrest is a sudden event, presumably due to electric instability of the heart, where evidence which allows other diagnosis' is lacking. If no resuscitation is applied or if resuscitation is unsuccessful, primary cardiac arrest is referred to as sudden death.2 Evidence of previous IHD may or may not be present. If death occurred in the absence of witnesses, the diagnosis is presumptive.


In [37]:
chunks[0]

KnowledgeChunk(doc_id='/home/hice1/jtamo3/bmed-sp-wang/Ben/Data/Medical-Cardiac-Knowledge/1979-nomenclature-and-criteria-for-diagnosis-of-ischemic-heart-disease-report-of-the-joint-international-society-and.txt::section-0::chunk-0', text='Nomenclature and Criteria for Diagnosis of Ischemic Heart Disease Report of the Joint International Society and Federation of Cardiology/World Health Organization Task Force on Standardization of Clinical Nomenclature LONG AGO, EPIDEMIOLOGISTS REALIZED the necessity of standardizing terminology and diagnostic criteria. Today, this need should also be recognized by clinicians. New, expensive medical and surgical methods of treatment are introduced every day, and it is essential to evaluate the effectiveness of these methods reliably and objectively. However, comparison of results is only possible if the object of the evaluation has been defined in a standard manner. The need for agreement on nomenclature is particularly urgent in the field of ischemic 

In [39]:
print(chunks[0].text)

Nomenclature and Criteria for Diagnosis of Ischemic Heart Disease Report of the Joint International Society and Federation of Cardiology/World Health Organization Task Force on Standardization of Clinical Nomenclature LONG AGO, EPIDEMIOLOGISTS REALIZED the necessity of standardizing terminology and diagnostic criteria. Today, this need should also be recognized by clinicians. New, expensive medical and surgical methods of treatment are introduced every day, and it is essential to evaluate the effectiveness of these methods reliably and objectively. However, comparison of results is only possible if the object of the evaluation has been defined in a standard manner. The need for agreement on nomenclature is particularly urgent in the field of ischemic heart disease (IHD) because extraordinary advances have been achieved by new methods of diagnosis and treatment which are a matter of worldwide interest and discussion. The task of this group is to propose an internationally acceptable nom

### Chunk length statistics
    Quick summary of tokenized word counts per chunk.

In [ ]:
word_counts = [len(chunk.text.split()) for chunk in chunks]
print({
    'min_words': min(word_counts),
    'max_words': max(word_counts),
    'mean_words': round(statistics.mean(word_counts), 2),
    'median_words': statistics.median(word_counts),
})

Optional: visualize the distribution. This cell will no-op if `matplotlib` is missing.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    print('matplotlib not installed; skipping histogram')
else:
    plt.hist(word_counts, bins=20, color='steelblue', edgecolor='black')
    plt.title('Chunk word counts')
    plt.xlabel('Words')
    plt.ylabel('Frequency')
    plt.show()

### Inspect a few chunks
    Helpful when tuning overlaps to ensure sections stay coherent.

In [ ]:
for chunk in chunks[:3]:
    print(f"ID: {chunk.doc_id}")
    print(f"Section: {chunk.metadata.get('section_title')} (index {chunk.metadata.get('section_index')})")
    print(textwrap.fill(chunk.text, width=100))
    print('-' * 80)

### Export the chunked corpus to JSONL
    Uncomment the cell below to write a JSONL file that the retrieval pipeline can ingest.

In [ ]:
# output_path = knowledge_dir / 'chunked_guidelines.jsonl'
# documents = [chunk.to_document() for chunk in chunks]
# export_documents_jsonl(documents, output_path)
# print(f'Wrote {len(documents)} chunks to {output_path}')

### Reload and sanity check the JSONL output
    This can help confirm the exported structure matches what the retriever expects.

In [ ]:
# if 'output_path' in globals():
#     reloaded: list[Document] = []
#     with open(output_path, 'r', encoding='utf-8') as handle:
#         for line in handle:
#             payload = json.loads(line)
#             reloaded.append(Document(**payload))
#     print(f'Reloaded {len(reloaded)} documents; first entry:')
#     print(reloaded[0])